# Lab 1 — Prompt Chaining + Validation Gates (§19.1.1)

The **Cogito Learn** teaching assistant will summarize a paper's abstract in Thai for first-year students
with a **chain** of LLM calls in a fixed order, with **gates** checking quality between steps

```
text → 🚦 input gate (LLM): is it an abstract?
     → ① extract key points → 🚦 gate 1 (code): exactly 3 points?
     → ② write a Thai summary → 🚦 gate 2 (code): at most 5 lines?
```

**What you'll learn**
1. Call an LLM through an API and ask for the answer as JSON (structured output)
2. Write gates both as plain code and with an LLM
3. Link the steps into a chain that stops as soon as a gate fails, with every intermediate result open to inspect

The data in this lab (the abstract and the announcement) is **simulated data written for teaching**; every model answer comes from a real run.

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §19.1.1 (p.369) — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **6 places marked `✏️ TODO`** (replace `___` or `...` with your code). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`19-1-lab1-chaining.en.ipynb`** (the same code as in the video).

## 0. Set up before you start

| What you need | Details |
|---|---|
| Python 3.10 or later | or use Google Colab, which already has it |
| OpenRouter API key | sign up at openrouter.ai → Keys → Create Key |
| Packages | `openai`, `python-dotenv` (installed in the next cell) |

**Add your API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file in the same folder as the notebook and add the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key directly in your code, and never share the `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

## 1. Calling the LLM through OpenRouter

We use **Gemma 4 26B** (open weights) through OpenRouter, which offers an *OpenAI-compatible* API
Every step of the chain calls the LLM through a single function, `ask_llm()`

In [ ]:
import os, json
from openai import OpenAI

In [ ]:
def load_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv()
        return os.environ["OPENROUTER_API_KEY"]

In [ ]:
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"provider": {"require_parameters": True}, "reasoning": {"enabled": False}}

In [ ]:
def ask_llm(prompt, temperature=0.7, schema=None):
    kw = {"response_format": {"type": "json_schema", "json_schema": {"name": "out", "strict": True, "schema": schema}}} if schema else {}
    resp = client.chat.completions.create(model=MODEL, extra_body=EXTRA, temperature=temperature,
                                          messages=[{"role": "user", "content": prompt}], **kw)
    text = resp.choices[0].message.content
    return json.loads(text) if schema else text

In [ ]:
print(ask_llm("Explain prompt chaining in one sentence.", temperature=0))

## 2. Test data (simulated)

- `ABSTRACT` — the abstract of a fictional paper about a retrieval-augmented teaching assistant
- `NOTICE` — a course announcement, which is **not** an abstract (to test that a gate can stop the chain)

In [ ]:
ABSTRACT = """We study whether a retrieval-augmented teaching assistant helps first-year students debug
Python assignments. In a 10-week course with 240 students, half used an assistant that first retrieves
the course's lecture notes, and half used the same LLM without retrieval. The retrieval group fixed 31%
more bugs on their own and asked human TAs 40% fewer repeated questions, but the benefit disappeared
for advanced students. We release anonymised interaction logs for further research."""

NOTICE = """Reminder for CS101 students: the midterm exam has moved from Room 402 to the main hall.
Please bring your student ID card and arrive 15 minutes early. Office hours this week are cancelled."""

## 3. Validation gates

Textbook: gates between steps **catch errors early**, before they flow into the next step, and can be either **programmatic** (code) or **LLM-based**

Every gate returns the same shape: `(ok, why)` — did it pass, and why

In [ ]:
# ✏️ TODO 1: Code gate: pass when there are exactly 3 key points
def gate_points(points):
    return ___, f"key points: {len(points)} (need 3)"

# ✏️ TODO 2: Code gate: count the summary's lines, and pass when there are no more than max_lines

def gate_lines(text, max_lines=5):
    n = ___
    return ___, f"lines: {n} (limit {max_lines})"

In [ ]:
print(gate_points(["problem", "method", "finding"]))
print(gate_points(["only one point"]))
print(gate_lines("1\n2\n3\n4\n5\n6"))

In [ ]:
IS_ABSTRACT = {"type": "object", "additionalProperties": False, "required": ["is_abstract", "reason"],
               "properties": {"is_abstract": {"type": "boolean"}, "reason": {"type": "string"}}}

# ✏️ TODO 3: LLM gate: ask with temperature 0 and the schema, then return (ok, why) from the dict you get

def gate_is_abstract(text):
    check = ask_llm("Is the following text the abstract of a research paper? Give a one-sentence reason.\n\n"
                    + text, temperature=___, schema=___)
    return ___, ___

## 4. Chain steps — each step is a separate LLM call

| Step | Task | temperature | Why |
|---|---|---|---|
| gate | is it an abstract? | 0 | steady judgement |
| ① | extract 3 points (JSON) | 0.2 | faithful to the source |
| ② | write the Thai summary | 0.7 | natural language |

In [ ]:
POINTS = {"type": "object", "additionalProperties": False, "required": ["points"],
          "properties": {"points": {"type": "array", "minItems": 1, "items": {"type": "string"}}}}

def extract_points(abstract):
    prompt = ("Extract exactly 3 key points from this research abstract: the problem, the method "
              "and the main finding. One short English sentence each. Use only facts stated in the text: "
              "no new adjectives, judgements or claims.\n\n" + abstract)
    return ask_llm(prompt, temperature=0.2, schema=POINTS)["points"]

# ✏️ TODO 4: Join the output of step one (the list of points) into text, one per line, each starting with "- "

def write_thai(points):
    bullets = ___
    prompt = ("Write a Thai summary for first-year university students using only these key points. "
              "Do not add any fact, adjective or claim that is not in the points. "
              "At most 5 lines, one sentence per line, no blank lines, keep technical terms in English. "
              "Return only the summary.\n\n" + bullets)
    return ask_llm(prompt, temperature=0.7)

## 5. Linking it into a chain

The system (not the model) fixes the order: **gate → ① → gate → ② → gate** — if any gate fails, the chain stops right away

In [ ]:
def gate(name, result):
    ok, why = result
    print(f"🚦 {name}: {'✅ pass' if ok else '🛑 STOP'} — {why}")
    return ok

In [ ]:
# ✏️ TODO 5: If the first gate (LLM) fails, stop the chain at once by returning None
def summarize_chain(text, max_lines=5):
    if not gate("input gate · research abstract? (LLM)", ___):
        ___

    # ✏️ TODO 6: Step one: extract the points, then have gate_points check them; if it fails, stop the chain

    points = ___
    print("① key points:", *points, sep="\n   • ")
    if not gate("gate 1 · exactly 3 points? (code)", ___):
        return None

    summary = write_thai(points)
    print("② Thai summary:\n" + summary)
    if not gate(f"gate 2 · at most {max_lines} lines? (code)", gate_lines(summary, max_lines)):
        return None
    return summary

In [ ]:
result = summarize_chain(ABSTRACT)

In [ ]:
result = summarize_chain(NOTICE)
print("result:", result)

In [ ]:
result = summarize_chain(ABSTRACT, max_lines=2)
print("result:", result)

## Summary

| What the textbook says (§19.1.1) | In the code |
|---|---|
| A fixed sequence of LLM calls | `summarize_chain()` — gate → ① → gate → ② → gate |
| Each step's output is the next step's context | `write_thai(points)` sees only the points from step one |
| Gates catch errors early | `NOTICE` stops at the input gate → steps ① ② never run · `max_lines=2` stops at gate 2 |
| A gate can be an LLM or code | `gate_is_abstract()` (LLM) · `gate_points()`, `gate_lines()` (code) |
| Each step uses a different temperature | 0 · 0.2 · 0.7 |
| Intermediate results are inspectable | `print()` at every step |

**Try next:** add a step ③ where the LLM writes a Thai title of at most 10 words, with a gate that checks the word count